In [1]:
import pandas as pd
import numpy as np
import simfin as sf

In [2]:
df = pd.read_csv("/Users/yashwanthkumar/Desktop/Desktop/ML project/early_layoff_warning/data/layoffs_cleaned.csv")
df.head(10)

,Company,Location_HQ,Industry,Laid_Off_Count,Date,Source,Funds_Raised,Stage,Date_Added,Country,Percentage,List_of_Employees_Laid_Off,Laid_Off_Count_Log
0,Oda,Oslo,Food,150.0,2024-06-05,https://techcrunch.com/2024/06/05/softbank-bac...,691.0,Unknown,2024-06-05 18:01:25,Norway,NaN,Unknown,5.017280
1,Pagaya,Tel Aviv,Finance,100.0,2024-06-05,https://www.calcalistech.com/ctechnews/article...,2000.0,Post-IPO,2024-06-05 23:11:24,Israel,0.20,Unknown,4.615121
2,Aleph Farms,Tel Aviv,Food,30.0,2024-06-05,https://www.calcalistech.com/ctechnews/article...,119.0,Unknown,2024-06-05 23:13:43,Israel,0.30,Unknown,3.433987
3,MoonPay,Dover,Crypto,30.0,2024-06-05,https://www.theblock.co/post/298638/moonpay-la...,651.0,Unknown,2024-06-05 23:12:47,United States,0.10,Unknown,3.433987
4,Microsoft,Seattle,Other,1000.0,2024-06-03,https://www.theverge.com/2024/6/3/24170902/mic...,1.0,Post-IPO,2024-06-03 20:27:12,United States,NaN,Unknown,6.908755
5,OrCam,Jerusalem,Healthcare,100.0,2024-06-03,https://www.calcalistech.com/ctechnews/article...,86.0,Unknown,2024-06-04 03:47:34,Israel,0.50,Unknown,4.615121
6,Google,SF Bay Area,Consumer,100.0,2024-05-31,https://www.businessinsider.com/google-cloud-l...,26.0,Post-IPO,2024-06-01 18:35:17,United States,NaN,Unknown,4.615121
7,Tropic,New York City,Finance,40.0,2024-05-31,https://www.linkedin.com/feed/update/urn:li:ac...,67.0,Series B,2024-06-01 18:33:50,United States,NaN,Unknown,3.713572
8,FlightStats,Portland,Travel,73.0,2024-05-30,https://www.oregonlive.com/silicon-forest/2024...,3.0,Acquired,2024-05-31 10:51:22,United States,NaN,Unknown,4.304065
9,ICANN,Los Angeles,Infrastructure,33.0,2024-05-30,https://www.icann.org/en/blogs/details/organiz...,172.5,Unknown,2024-06-05 18:07:23,United States,0.07,Unknown,3.526361


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv('SIMFIN_API_KEY')
print(f"Key loaded: {api_key[:4]}...")

sf.set_api_key(api_key)
sf.set_data_dir('data/simfin/')

# Test - load quarterly income statements
df_income = sf.load_income(variant='quarterly', market='us')
print(df_income.shape)
print(df_income.head())

Key loaded: cd0a...
Dataset "us-income-quarterly" on disk (0 days old).
- Loading from disk ... Done!
(47760, 26)
                    SimFinId Currency  Fiscal Year Fiscal Period Publish Date  \
Ticker Report Date                                                              
A      2020-07-31      45846      USD         2020            Q3   2020-09-01   
       2020-10-31      45846      USD         2020            Q4   2020-12-18   
       2021-01-31      45846      USD         2021            Q1   2021-03-02   
       2021-04-30      45846      USD         2021            Q2   2021-06-01   
       2021-07-31      45846      USD         2021            Q3   2021-09-01   

                   Restated Date  Shares (Basic)  Shares (Diluted)  \
Ticker Report Date                                                   
A      2020-07-31     2021-09-01     309000000.0       312000000.0   
       2020-10-31     2021-09-01     308000000.0       311000000.0   
       2021-01-31     2022-03-03     3

In [4]:
df_balance = sf.load_balance(variant='quarterly', market='us')
print(df_balance.shape)
print(df_balance.columns.tolist())

Dataset "us-balance-quarterly" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
(47763, 28)
['SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Publish Date', 'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Cash, Cash Equivalents & Short Term Investments', 'Accounts & Notes Receivable', 'Inventories', 'Total Current Assets', 'Property, Plant & Equipment, Net', 'Long Term Investments & Receivables', 'Other Long Term Assets', 'Total Noncurrent Assets', 'Total Assets', 'Payables & Accruals', 'Short Term Debt', 'Total Current Liabilities', 'Long Term Debt', 'Total Noncurrent Liabilities', 'Total Liabilities', 'Share Capital & Additional Paid-In Capital', 'Treasury Stock', 'Retained Earnings', 'Total Equity', 'Total Liabilities & Equity']


In [8]:
# Reset index to make Ticker and Report Date regular columns
df_income_clean = df_income[['Revenue', 'Net Income']].copy().reset_index()
df_balance_clean = df_balance[['Cash, Cash Equivalents & Short Term Investments']].copy().reset_index()

# Rename for clarity
df_balance_clean = df_balance_clean.rename(columns={
    'Cash, Cash Equivalents & Short Term Investments': 'Cash_Reserves'
})

# Merge on Ticker + Report Date
df_simfin = df_income_clean.merge(df_balance_clean, on=['Ticker', 'Report Date'], how='left')

print(df_simfin.shape)
print(df_simfin.head())

(47760, 5)
  Ticker Report Date       Revenue  Net Income  Cash_Reserves
0      A  2020-07-31  1.261000e+09   199000000   1.358000e+09
1      A  2020-10-31  1.483000e+09   222000000   1.441000e+09
2      A  2021-01-31  1.548000e+09   288000000   1.329000e+09
3      A  2021-04-30  1.525000e+09   216000000   1.380000e+09
4      A  2021-07-31  1.586000e+09   264000000   1.428000e+09


In [14]:
df_simfin = df_simfin.sort_values(["Ticker", "Report Date"])

df_simfin["Profit_Margin"] = df_simfin["Net Income"] / df_simfin["Revenue"]

df_simfin["Revenue_Growth"] = df_simfin.groupby("Ticker")["Revenue"].pct_change()

df_simfin = df_simfin.drop(columns=['Revenue', 'Net Income'])


print(df_simfin.shape)
print(df_simfin.head(10))
print(df_simfin.isnull().sum())



(47760, 5)
  Ticker Report Date  Cash_Reserves  Profit_Margin  Revenue_Growth
0      A  2020-07-31   1.358000e+09       0.157811             NaN
1      A  2020-10-31   1.441000e+09       0.149697        0.176051
2      A  2021-01-31   1.329000e+09       0.186047        0.043830
3      A  2021-04-30   1.380000e+09       0.141639       -0.014858
4      A  2021-07-31   1.428000e+09       0.166456        0.040000
5      A  2021-10-31   1.575000e+09       0.266265        0.046658
6      A  2022-01-31   1.158000e+09       0.169056        0.008434
7      A  2022-04-30   1.207000e+09       0.170504       -0.040024
8      A  2022-07-31   1.053000e+09       0.191502        0.069073
9      A  2022-10-31   1.053000e+09       0.199027        0.076251
Ticker               0
Report Date          0
Cash_Reserves      210
Profit_Margin     5218
Revenue_Growth    8698
dtype: int64


In [18]:
df["Date"] = pd.to_datetime(df["Date"])

df_simfin["Report Date"] = pd.to_datetime(df_simfin["Report Date"])

mapping_df = pd.read_csv("/Users/yashwanthkumar/Desktop/Desktop/ML project/early_layoff_warning/data/ticker_mapping.csv")

df = df.merge(mapping_df, on="Company", how="left")

print(f"Rows with ticker: {df['Ticker'].notna().sum()}")
print(f"Rows without ticker: {df['Ticker'].isna().sum()}")


Rows with ticker: 480
Rows without ticker: 1909


In [19]:
def get_financials_before_layoff(row, df_fin):
    # Private companies - no ticker
    if pd.isna(row['Ticker']):
        return pd.Series({
            'Cash_Reserves': np.nan,
            'Profit_Margin': np.nan,
            'Revenue_Growth': np.nan
        })
    
    # Get all quarters for this ticker
    ticker_data = df_fin[df_fin['Ticker'] == row['Ticker']]
    
    # Find quarters strictly before layoff date
    before = ticker_data[ticker_data['Report Date'] < row['Date']]
    
    if before.empty:
        return pd.Series({
            'Cash_Reserves': np.nan,
            'Profit_Margin': np.nan,
            'Revenue_Growth': np.nan
        })
    
    # Take the most recent quarter before layoff
    closest = before.sort_values('Report Date').iloc[-1]
    
    return pd.Series({
        'Cash_Reserves': closest['Cash_Reserves'],
        'Profit_Margin': closest['Profit_Margin'],
        'Revenue_Growth': closest['Revenue_Growth']
    })

# Apply - takes 1-2 minutes
print("Merging financials... takes a minute")
financial_cols = df.apply(
    lambda row: get_financials_before_layoff(row, df_simfin),
    axis=1
)

df = pd.concat([df, financial_cols], axis=1)

print(f"Shape: {df.shape}")
print(df[['Company', 'Ticker', 'Date', 'Cash_Reserves', 'Profit_Margin', 'Revenue_Growth']].head(10))

Merging financials... takes a minute
Shape: (2389, 19)
       Company Ticker       Date  Cash_Reserves  Profit_Margin  Revenue_Growth
0          Oda    NaN 2024-06-05            NaN            NaN             NaN
1       Pagaya    NaN 2024-06-05            NaN            NaN             NaN
2  Aleph Farms    NaN 2024-06-05            NaN            NaN             NaN
3      MoonPay    NaN 2024-06-05            NaN            NaN             NaN
4    Microsoft   MSFT 2024-06-03   8.002100e+10       0.354667       -0.002612
5        OrCam    NaN 2024-06-03            NaN            NaN             NaN
6       Google   GOOG 2024-05-31   1.080900e+11       0.293796       -0.066864
7       Tropic    NaN 2024-05-31            NaN            NaN             NaN
8  FlightStats    NaN 2024-05-30            NaN            NaN             NaN
9        ICANN    NaN 2024-05-30            NaN            NaN             NaN


In [20]:
print(f"Rows with Cash_Reserves: {df['Cash_Reserves'].notna().sum()}")
print(f"Rows with Profit_Margin: {df['Profit_Margin'].notna().sum()}")
print(f"Rows with Revenue_Growth: {df['Revenue_Growth'].notna().sum()}")
print(f"Total rows: {df.shape[0]}")

Rows with Cash_Reserves: 262
Rows with Profit_Margin: 259
Rows with Revenue_Growth: 255
Total rows: 2389


In [21]:
df['Laid_Off'] = 1
print(df['Laid_Off'].value_counts())

Laid_Off
1    2389
Name: count, dtype: int64


In [32]:
# Find tech companies from our own ticker mapping that never laid off
# These are similar companies to our layoff dataset - same industry, same type
all_mapped_tickers = mapping_df['Ticker'].tolist()
layoff_tickers = df['Ticker'].dropna().unique().tolist()

# Tickers that were mapped but never appeared in layoffs dataset
never_laid_off = [t for t in all_mapped_tickers if t not in layoff_tickers]
print(f"Tech companies that never laid off: {len(never_laid_off)}")

# Get their simfin financials
never_laid_off_fin = df_simfin[df_simfin['Ticker'].isin(never_laid_off)].copy()
print(f"Simfin rows found: {never_laid_off_fin.shape}")
print(f"Unique tickers found: {never_laid_off_fin['Ticker'].nunique()}")

Tech companies that never laid off: 0
Simfin rows found: (0, 5)
Unique tickers found: 0


In [33]:
# Companies in layoffs dataset that have both a ticker AND simfin data
layoff_tickers = df['Ticker'].dropna().unique().tolist()
tickers_with_simfin = df_simfin[
    df_simfin['Ticker'].isin(layoff_tickers)
]['Ticker'].unique().tolist()

print(f"Layoff companies with simfin data: {len(tickers_with_simfin)}")
print(tickers_with_simfin[:20])


Layoff companies with simfin data: 164
['AAPL', 'ABNB', 'ABSI', 'ADBE', 'ADI', 'ADPT', 'ADSK', 'AFRM', 'AKAM', 'AMPL', 'AMZN', 'APP', 'APPF', 'ASAN', 'AYX', 'BARK', 'BCOV', 'BIRD', 'BMBL', 'BMRN']


In [34]:
from datetime import timedelta

all_rows = []

for ticker in tickers_with_simfin:
    # Get all layoff dates for this ticker
    layoff_dates = df[df['Ticker'] == ticker]['Date'].tolist()
    
    # Get all quarterly financials for this ticker
    ticker_fin = df_simfin[df_simfin['Ticker'] == ticker].copy()
    
    if ticker_fin.empty:
        continue
    
    # Get company info from layoffs dataset
    company_info = df[df['Ticker'] == ticker].iloc[0]
    
    for _, fin_row in ticker_fin.iterrows():
        quarter_date = fin_row['Report Date']
        
        # Check this quarter against all layoff dates for this company
        label = 0  # default - no layoff signal
        
        for layoff_date in layoff_dates:
            layoff_date = pd.to_datetime(layoff_date)
            
            # Skip quarters after layoff happened
            if quarter_date >= layoff_date:
                label = -1  # mark to skip
                break
            
            # Within 6 months before layoff = label 1
            six_months_before = layoff_date - timedelta(days=180)
            if quarter_date >= six_months_before:
                label = 1
                break
        
        # Skip quarters after layoff
        if label == -1:
            continue
        
        all_rows.append({
            'Ticker': ticker,
            'Company': company_info['Company'],
            'Industry': company_info['Industry'],
            'Stage': company_info['Stage'],
            'Country': company_info['Country'],
            'Funds_Raised': company_info['Funds_Raised'],
            'Quarter': quarter_date,
            'Cash_Reserves': fin_row['Cash_Reserves'],
            'Profit_Margin': fin_row['Profit_Margin'],
            'Revenue_Growth': fin_row['Revenue_Growth'],
            'Laid_Off': label
        })

df_timebased = pd.DataFrame(all_rows)

print(f"Total rows: {df_timebased.shape}")
print(f"\nLabel distribution:")
print(df_timebased['Laid_Off'].value_counts())
print(f"\nUnique companies: {df_timebased['Ticker'].nunique()}")

Total rows: (1301, 11)

Label distribution:
Laid_Off
0    871
1    430
Name: count, dtype: int64

Unique companies: 147


In [35]:
# Drop Ticker and Quarter - not model features
df_final = df_timebased.drop(columns=['Ticker', 'Quarter', 'Company'])

print(df_final.shape)
print(df_final.columns.tolist())
print(df_final.isnull().sum())

(1301, 8)
['Industry', 'Stage', 'Country', 'Funds_Raised', 'Cash_Reserves', 'Profit_Margin', 'Revenue_Growth', 'Laid_Off']
Industry            0
Stage               0
Country             0
Funds_Raised        0
Cash_Reserves      14
Profit_Margin      13
Revenue_Growth    138
Laid_Off            0
dtype: int64


In [37]:
# Fill nulls with median
for col in ['Cash_Reserves', 'Profit_Margin', 'Revenue_Growth']:
    median_val = df_final[col].median()
    df_final[col] = df_final[col].fillna(median_val)
    print(f"{col} median: {median_val:.4f}")

# Verify zero nulls
print("\nNull counts:")
print(df_final.isnull().sum())

# Save - overwrite old master dataset
df_final.to_csv('/Users/yashwanthkumar/Desktop/Desktop/ML project/early_layoff_warning/data/master_dataset.csv', index=False)
print(f"\nSaved master_dataset.csv with shape: {df_final.shape}")

Cash_Reserves median: 1014400000.0000
Profit_Margin median: -0.0648
Revenue_Growth median: 0.0451

Null counts:
Industry          0
Stage             0
Country           0
Funds_Raised      0
Cash_Reserves     0
Profit_Margin     0
Revenue_Growth    0
Laid_Off          0
dtype: int64

Saved master_dataset.csv with shape: (1301, 8)
